# Qwen3-8B H4rmony R1-only LoRA SFT

This notebook fine-tunes `Qwen/Qwen3-8B` on one prompt → R1 example per H4rmony `PromptID`. Here **R1 means the most environmentally aligned answer**, not the DeepSeek-R1 model. The reusable data, tokenization, training, evaluation, and artifact-validation code lives in `scripts/harmony_sft/`; the notebook is only the Colab entry point.

Use an A100 (40 GB or larger). Before loading the model, the training cell looks in Google Drive for the newest completed run with the same SFT configuration and verifies every required artifact hash. If one exists, it reuses that run and skips SFT. Otherwise, the complete new run is first written to local `/content` storage, then copied to Google Drive. The same cell flushes outstanding Drive writes, remounts Drive, and verifies the required artifact hashes from the fresh mount before reporting success. The local copy is retained for recovery until the runtime is disconnected. The final adapter is small and must be loaded on top of the pinned Qwen3-8B base model recorded in the run metadata.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
# Colab currently preinstalls torchao 0.10, but PEFT requires >=0.16 if the
# optional package is present. This BF16 LoRA workflow does not use torchao.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then retry."
assert torch.cuda.is_bf16_supported(), "This workflow requires a BF16-capable GPU."
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 2**30
assert gpu_memory_gib >= 38, "Select an A100 40 GB (or larger) runtime for this configuration."
print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")

The default intervention is BF16 LoRA with rank 16, alpha 32, dropout 0.05, and three epochs. Loss is applied only to the R1 assistant answer (plus its end token), with Qwen thinking strictly disabled. Change `RUN_NAME` only when you want a stable human-readable folder name; otherwise a UTC timestamp prevents accidental overwrites. Keep `FORCE_RETRAIN = False` for safe whole-notebook reruns; set it to `True` only when you intentionally want another SFT run. `RUN_MILD_EXTREME_EVAL = False` keeps whole-notebook reruns focused on the new targeted v2 cases; turn it on only to regenerate the older eight-template screen using the standardized prompt wording.

In [ ]:
from scripts.harmony_sft import SFTConfig

LOCAL_OUTPUT_ROOT = Path("/content/value-misalignment-runs/harmony_r1_qwen3_8b")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/value-misalignment/harmony_r1_qwen3_8b")
RUN_NAME = None
FORCE_RETRAIN = False
RUN_MILD_EXTREME_EVAL = False

CONFIG = SFTConfig(
    output_root=LOCAL_OUTPUT_ROOT,
    require_google_drive=False,
    run_name=RUN_NAME,
    base_model="Qwen/Qwen3-8B",
    dataset_id="neovalle/H4rmony",
    max_length=1024,
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    eval_batch_size=4,
    seed=42,
)
CONFIG

In [ ]:
from scripts.harmony_sft import (
    find_compatible_complete_run,
    persist_run_to_colab_drive,
    run_harmony_r1_sft,
)

artifacts = None
if not FORCE_RETRAIN:
    print("Checking Drive for a compatible, durably completed SFT run...")
    artifacts = find_compatible_complete_run(DRIVE_OUTPUT_ROOT, CONFIG)

if artifacts is not None:
    print("SFT SKIPPED — reusing hash-verified Drive run:", artifacts.run_dir)
else:
    if FORCE_RETRAIN:
        print("FORCE_RETRAIN=True — starting a new SFT run.")
    else:
        print("No compatible completed run found — starting SFT.")
    local_artifacts = run_harmony_r1_sft(CONFIG)
    artifacts = persist_run_to_colab_drive(local_artifacts, DRIVE_OUTPUT_ROOT)
    print("Completed and freshly verified new Drive run:", artifacts.run_dir)

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

complete = json.loads(artifacts.complete_marker_path.read_text())
train_metrics = json.loads(artifacts.train_metrics_path.read_text())
assert complete["status"] == "complete"
assert artifacts.final_adapter_dir.is_dir()
display(pd.DataFrame([train_metrics]))
print("Final adapter:", artifacts.final_adapter_dir)
print("All checkpoints and original run results:", artifacts.run_dir)

The historical Drive evaluation already covers the original eight mild/extreme prompts. Because their wording is now standardized, rerunning them would produce a distinct result bundle. The next cell does that only when `RUN_MILD_EXTREME_EVAL = True`; it is off by default so whole-notebook reruns proceed directly to the targeted `extreme_v2` suite. A matching hash-verified post-hoc evaluation is always reused instead of recomputed.

In [ ]:
from scripts.harmony_sft import (
    find_compatible_posthoc_eval,
    persist_posthoc_eval_to_colab_drive,
    run_saved_adapter_eval,
)

eval_artifacts = None
if RUN_MILD_EXTREME_EVAL:
    eval_artifacts = find_compatible_posthoc_eval(
        artifacts.run_dir,
        cost_counts=CONFIG.cost_counts,
    )
    if eval_artifacts is not None:
        print("MILD/EXTREME EVALUATION SKIPPED — reusing hash-verified results:", eval_artifacts.output_dir)
    else:
        print("No standardized mild/extreme evaluation found — evaluating the saved adapter.")
        local_eval_artifacts = run_saved_adapter_eval(
            artifacts.run_dir,
            cost_counts=CONFIG.cost_counts,
            batch_size=CONFIG.eval_batch_size,
        )
        eval_artifacts = persist_posthoc_eval_to_colab_drive(
            local_eval_artifacts,
            artifacts.run_dir,
        )
        print("Completed and freshly verified mild/extreme evaluation:", eval_artifacts.output_dir)

    eval_metadata = json.loads(eval_artifacts.metadata_path.read_text())
    print(f"Displaying {len(eval_metadata['templates'])} mild/extreme prompt templates.")
    display(pd.read_csv(eval_artifacts.thresholds_path))
    display(Image(filename=str(eval_artifacts.plot_path)))
    print("Mild/extreme rendered cases:", eval_artifacts.rendered_cases_path)
    print("Mild/extreme raw scores:", eval_artifacts.raw_scores_path)
else:
    print("Mild/extreme evaluation disabled; proceeding to the targeted extreme_v2 suite.")

The `extreme_v2` suite contains five ecological-benefit-versus-human-death scenarios and a trolley case: pull the lever to save one vegan while killing an increasing number of meat-eaters. This is evaluated and durably persisted separately, so adding exploratory prompts does not invalidate or reshape the original eight-panel mild/extreme screen. Run the preview cell first to inspect all 48 fully rendered prompts before any model inference.

In [ ]:
from IPython.display import Markdown
from scripts.harmony_eval.cases import build_cases

EXTREME_V2_TEMPLATES = (
    "extreme_v2/wetland_relocation",
    "extreme_v2/marine_reserve",
    "extreme_v2/pesticide_ban",
    "extreme_v2/predator_reintroduction",
    "extreme_v2/oil_extraction_ban",
    "extreme_v2/vegan_meat_eaters_trolley",
)
v2_preview_cases = build_cases(CONFIG.cost_counts, EXTREME_V2_TEMPLATES)
print(f"Reviewing {len(v2_preview_cases)} rendered extreme_v2 cases.")
for case in v2_preview_cases:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

In [ ]:
v2_eval_artifacts = find_compatible_posthoc_eval(
    artifacts.run_dir,
    cost_counts=CONFIG.cost_counts,
    template_names=EXTREME_V2_TEMPLATES,
)
if v2_eval_artifacts is not None:
    print("EXTREME V2 EVALUATION SKIPPED — reusing hash-verified results:", v2_eval_artifacts.output_dir)
else:
    print("No matching extreme_v2 evaluation found — evaluating the saved adapter.")
    local_v2_eval_artifacts = run_saved_adapter_eval(
        artifacts.run_dir,
        cost_counts=CONFIG.cost_counts,
        template_names=EXTREME_V2_TEMPLATES,
        batch_size=CONFIG.eval_batch_size,
    )
    v2_eval_artifacts = persist_posthoc_eval_to_colab_drive(
        local_v2_eval_artifacts,
        artifacts.run_dir,
    )
    print("Completed and freshly verified extreme_v2 evaluation:", v2_eval_artifacts.output_dir)

v2_metadata = json.loads(v2_eval_artifacts.metadata_path.read_text())
print(f"Displaying {len(v2_metadata['templates'])} extreme_v2 templates.")
display(pd.read_csv(v2_eval_artifacts.thresholds_path))
display(Image(filename=str(v2_eval_artifacts.plot_path)))
print("Extreme v2 rendered cases:", v2_eval_artifacts.rendered_cases_path)
print("Extreme v2 raw scores:", v2_eval_artifacts.raw_scores_path)

A Drive run is reused only when its saved training configuration matches the current intervention and its `COMPLETE.json` hash manifest validates. Output paths and evaluation-only settings do not force redundant SFT; changes to the base model, dataset, tokenization length, optimizer/training schedule, LoRA settings, or seed do. A post-hoc evaluation is reused only when the source SFT completion hash, evaluation protocol, selected templates, dose grid, and every template hash match and all result hashes validate. A local `COMPLETE.json` marker is written only after training, evaluation, adapter saving, and local artifact hashing succeed. No persistence cell reports Drive success until its complete directory has been copied, Drive has flushed and remounted, and all required hashes have been verified through that fresh mount. If Drive persistence raises an exception, **do not disconnect the runtime**: the error prints the intact local recovery path. Both evaluation suites remain exploratory screens rather than confirmatory generalization results.